In [ ]:
import pika
import sys

up_ime = input("Uporabniško ime: ")
len_user = len(up_ime)

credentials = pika.PlainCredentials('martin', 'martin00')

parameters =  pika.ConnectionParameters('149.62.71.186', credentials=credentials)

connection = pika.BlockingConnection(parameters)

channel = connection.channel()

channel.exchange_declare(exchange='logs', exchange_type='fanout')

result = channel.queue_declare(queue='', exclusive=True)
queue_name = result.method.queue

channel.queue_bind(exchange='logs', queue=queue_name)


while True:
    
    while True:
        received = channel.basic_get(queue_name)[2]
        if received != None:
            decoded = received.decode()
            if decoded[:len_user] != up_ime:
                print(decoded)
        else:
            break
        
    
    message_content = input(up_ime + ":")

    message = up_ime + ': ' + message_content
    channel.basic_publish(exchange='logs', routing_key='', body=message)




Let's break down `received = channel.basic_get(queue_name)[2]` step-by-step:

1.  **`channel.basic_get(queue_name)`:**

    * This is a method call to the `basic_get` function on the `channel` object (which represents a RabbitMQ channel in your Python `pika` code).
    * `queue_name` is a variable that holds the name of the RabbitMQ queue you want to retrieve a message from.
    * `basic_get` is a method that retrieves a *single* message from the specified queue.
    * Crucially, `basic_get` is a *polling* operation. It checks the queue right now, and if there's a message, it gets it. If not, it returns immediately.
    * Unlike `basic_consume`, which sets up a continuous consumer, `basic_get` is a one-time retrieval.

2.  **Return Value of `channel.basic_get()`:**

    * The `channel.basic_get()` method returns a tuple with three elements:
        * `method_frame`: A `pika.spec.Deliver` object containing metadata about the message delivery (e.g., delivery tag, exchange, routing key).
        * `header_frame`: A `pika.spec.BasicProperties` object containing message properties (e.g., content type, headers).
        * `body`: The actual message content, which is a `bytes` object.
    * If the queue is empty (no message is available), `basic_get()` returns `(None, None, None)`.

3.  **`[2]`:**

    * This is an index operator that accesses the element at index 2 of the tuple returned by `channel.basic_get()`.
    * Since Python uses zero-based indexing, `[2]` refers to the *third* element of the tuple.
    * Therefore, `channel.basic_get(queue_name)[2]` extracts the `body` (message content) from the tuple.

**In simpler terms:**

"Go to the queue named `queue_name`, get one message, and give me the message content (the body) of that message."

**Example:**

```python
    import pika

    # ... (connection and channel setup)

    received = channel.basic_get('my_queue')[2]

    if received is not None:
        message_body = received.decode('utf-8')  # Decode the bytes
        print(f"Received message: {message_body}")
        # Process the message
    else:
        print("No message in the queue.")
```
